# Baseline walkthrough: Ca Mau, Sentinel-1D, 2026-09-29

Review notebook for the CA-CFAR baseline. It reads the committed outputs only; nothing heavy runs here.

> "Dark" does not mean illegal. No AIS was connected for this run, so nothing is labeled dark.

In [ ]:
import json
import geopandas as gpd
import pandas as pd
import pyogrio
from pathlib import Path

DATA = Path('..') / 'data'
print(json.dumps(json.loads((DATA / 's1_search_summary.json').read_text()), indent=1)[:1500])

## 1. Scenes over the AOI in the last 90 days

In [ ]:
scenes = pd.read_csv(DATA / 's1_scenes.csv', parse_dates=['start_utc'])
scenes.groupby(['mission', 'pass_dir', 'orbit_rel']).agg(n=('product_id', 'size'), mean_cover=('aoi_coverage', 'mean')).round(2)

In [ ]:
fp = gpd.read_file(DATA / 's1_footprints.gpkg', layer='s1_footprints_4326')
aoi = gpd.read_file(DATA / 'aoi.gpkg', layer='aoi_4326')
ax = fp.boundary.plot(color='tab:orange', linewidth=0.5, figsize=(7, 6))
aoi.boundary.plot(ax=ax, color='black')
ax.set_title('Sentinel-1D footprints vs AOI');

## 2. Detections

Classes: `high` = VV and VH, `medium` = one channel, `fixed` = recurs on both earlier dates, `low` = likely sea clutter. The `about` table holds every setting and the full caveat.

In [ ]:
print(pyogrio.list_layers(DATA / 'detections_baseline.gpkg'))
about = pyogrio.read_dataframe(DATA / 'detections_baseline.gpkg', layer='about').T
about

In [ ]:
det = gpd.read_file(DATA / 'detections_baseline.gpkg', layer='detections_baseline_utm48n')
det.confidence.value_counts()

In [ ]:
vessels = det[det.confidence.isin(['high', 'medium'])]
vessels[['length_est_m', 'scr_vv_db', 'scr_vh_db', 'inc_angle_deg']].describe().round(1)

In [ ]:
ax = det[det.confidence == 'low'].plot(markersize=1, color='0.6', figsize=(8, 8))
for cls, c in [('fixed', '#1baf7a'), ('medium', '#eb6834'), ('high', '#2a78d6')]:
    det[det.confidence == cls].plot(ax=ax, markersize=8, color=c, label=cls)
ax.legend(); ax.set_title('Detections by class (UTM 48N)');

## 3. Things to check by eye (in ArcGIS Pro or here)

1. Do the `fixed` rows near the east coast line up with known wind farms or stake-net lines?
2. Pick 20 `medium` detections: vessel, structure, or clutter? This is the start of a 1D ground-truth set.
3. Where do `low` objects cluster? Compare with the VV raster `data/outputs/small/sigma0_vv_db_utm48n_40m_u8.tif`.